# Importing Libraries

In [1]:
# Installing necessary packages
!pip install pandas numpy matplotlib seaborn scipy scikit-learn plotly

# Importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import scipy.stats as stats

print("Libraries imported successfully!")

Libraries imported successfully!


# Uploading Dataset

In [2]:
from google.colab import files
uploaded = files.upload()

import pandas as pd
import io

filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))

print("Data imported successfully!")
print(f"Shape: {df.shape}")
df.head()

Saving season-2425.csv to season-2425.csv
Data imported successfully!
Shape: (380, 22)


,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,...,HST,AST,HF,AF,HC,AC,HY,AY,HR,AR
0,16/08/24,Man United,Fulham,1,0,H,0,0,D,R Jones,...,5,2,12,10,7,8,2,3,0,0
1,17/08/24,Ipswich,Liverpool,0,2,A,0,0,D,T Robinson,...,2,5,9,18,2,10,3,1,0,0
2,17/08/24,Arsenal,Wolves,2,0,H,1,0,H,J Gillett,...,6,3,17,14,8,2,2,2,0,0
3,17/08/24,Everton,Brighton,0,3,A,0,1,A,S Hooper,...,1,5,8,8,1,5,1,1,1,0
4,17/08/24,Newcastle,Southampton,1,0,H,1,0,H,C Pawson,...,1,4,15,16,3,12,2,4,1,0


# Statistical Analysis of Performance Metrics

In [3]:
import pandas as pd
from scipy import stats

try:
    # --- Step 1: Load Data ---
    file_path = 'season-2425.csv'
    df = pd.read_csv(file_path)

    # --- Step 2: Data Transformation (Calculate League Table) ---

    # Get all unique team names
    teams = pd.concat([df['HomeTeam'], df['AwayTeam']]).unique()

    # Initialize dictionaries to store points and goal difference
    points = {team: 0 for team in teams}
    gd = {team: 0 for team in teams}

    # Iterate through each match to calculate points and GD
    for index, row in df.iterrows():
        home_team = row['HomeTeam']
        away_team = row['AwayTeam']
        home_goals = row['FTHG']
        away_goals = row['FTAG']

        # Calculate Goal Difference
        gd[home_team] += home_goals - away_goals
        gd[away_team] += away_goals - home_goals

        # Assign Points
        if row['FTR'] == 'H':
            points[home_team] += 3
        elif row['FTR'] == 'A':
            points[away_team] += 3
        elif row['FTR'] == 'D':
            points[home_team] += 1
            points[away_team] += 1

    # Create League Table DataFrame
    league_table = pd.DataFrame({'Points': points, 'GD': gd})

    # Sort by Points (desc), then GD (desc)
    league_table = league_table.sort_values(by=['Points', 'GD'], ascending=[False, False])

    # Add League Position column
    league_table['League_Position'] = range(1, len(league_table) + 1)

    # --- Step 3: Data Transformation (Aggregate Team Stats) ---

    # Calculate Total Clean Sheets per team
    home_cs = df[df['FTAG'] == 0].groupby('HomeTeam').size().rename('Home_CS')
    away_cs = df[df['FTHG'] == 0].groupby('AwayTeam').size().rename('Away_CS')
    total_cs = pd.concat([home_cs, away_cs], axis=1).fillna(0).sum(axis=1).rename('Total_Clean_Sheets')

    # Calculate Total Cards per team (Yellow + Red)
    home_cards = (df.groupby('HomeTeam')['HY'].sum() + df.groupby('HomeTeam')['HR'].sum()).rename('Home_Cards')
    away_cards = (df.groupby('AwayTeam')['AY'].sum() + df.groupby('AwayTeam')['AR'].sum()).rename('Away_Cards')
    total_cards = pd.concat([home_cards, away_cards], axis=1).fillna(0).sum(axis=1).rename('Total_Cards')

    # --- Step 4: Merge Aggregated Data ---

    # Merge all stats into one DataFrame, indexed by team name
    team_stats = league_table.merge(total_cs, left_index=True, right_index=True)
    team_stats = team_stats.merge(total_cards, left_index=True, right_index=True)

    # --- Step 5: Run Statistical Tests ---

    # Test 1: Clean Sheets vs. League Position
    cs_corr = stats.pearsonr(team_stats['Total_Clean_Sheets'], team_stats['League_Position'])

    # Test 2: Total Cards vs. League Position
    cards_corr = stats.pearsonr(team_stats['Total_Cards'], team_stats['League_Position'])

    # Test 3: Home Advantage (Paired t-test on the original 380-match dataframe)
    home_adv = stats.ttest_rel(df['FTHG'], df['FTAG'])

    # --- Step 6: Print Results ---

    print("--- Aggregated Team Stats (Top 5) ---")
    print(team_stats.head())
    print("\n")

    print("--- Statistical Test Results (for your paper) ---")
    print(f"1. Clean Sheets vs. League Position: r = {cs_corr[0]:.4f}, p = {cs_corr[1]:.4f}")
    print(f"2. Total Cards vs. League Position:     r = {cards_corr[0]:.4f}, p = {cards_corr[1]:.4f}")
    print(f"3. Home Advantage (Home vs. Away Goals): t = {home_adv.statistic:.4f}, p = {home_adv.pvalue:.4f}")

except FileNotFoundError:
    print(f"Error: The file 'season-2425.csv' was not found.")
except KeyError as e:
    print(f"Error: A required column {e} was not found in the CSV.")
    print("Please ensure your CSV contains columns like 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'HY', 'AY', 'HR', 'AR'.")

--- Aggregated Team Stats (Top 5) ---
           Points  GD  League_Position  Total_Clean_Sheets  Total_Cards
Liverpool      84  45                1                14.0           67
Arsenal        74  35                2                13.0           70
Man City       71  28                3                13.0           59
Chelsea        69  21                4                11.0          100
Newcastle      66  21                5                13.0           69


--- Statistical Test Results (for your paper) ---
1. Clean Sheets vs. League Position: r = -0.8132, p = 0.0000
2. Total Cards vs. League Position:     r = 0.3770, p = 0.1013
3. Home Advantage (Home vs. Away Goals): t = 0.9622, p = 0.3366
